PREDICTIONS & EVALUATION

In [50]:
#Import necessary libraries

import pandas as pd
import joblib

# Load saved model
model = joblib.load("../models/epl_random_forest_model.pkl")

print("Model loaded.")

Model loaded.


In [51]:
# Load future-prediction dataset

df = pd.read_csv("../data/cleaned/epl-ml-ready.csv")

print(df.head())

      Team   Season  PrevPos  PrevPts  PrevGD  PrevGF  PrevGA  PrevWinPct  \
0  Arsenal  2001-02      2.0     70.0    25.0    63.0    38.0    0.526316   
1  Arsenal  2002-03      1.0     87.0    43.0    79.0    36.0    0.684211   
2  Arsenal  2003-04      2.0     78.0    43.0    85.0    42.0    0.605263   
3  Arsenal  2004-05      1.0     90.0    47.0    73.0    26.0    0.684211   
4  Arsenal  2005-06      2.0     83.0    51.0    87.0    36.0    0.657895   

   PrevDrawPct  PrevLossPct  Pos  
0     0.263158     0.210526    1  
1     0.236842     0.078947    2  
2     0.236842     0.157895    1  
3     0.315789     0.000000    2  
4     0.210526     0.131579    4  


In [52]:
# Previous-season predictive features

features = [
    "PrevPos",
    "PrevPts",
    "PrevGD",
    "PrevGF",
    "PrevGA",
    "PrevWinPct",
    "PrevDrawPct",
    "PrevLossPct"
]

# Create prediction inputs
x = df[features]

# Generate predictions
predicted_positions = model.predict(x[model.feature_names_in_])

# Clean predictions
predicted_positions = predicted_positions.round().clip(1, 20)

# Add predictions to dataframe
df["PredictedPosition"] = predicted_positions

In [53]:
# Compare actual vs predicted

comparison = df[[
    "Team",
    "Season",
    "Pos",
    "PredictedPosition"
]]

for season, season_table in comparison.groupby("Season"):
    print(f"\nSeason: {season}")
    
    season_table = season_table.sort_values(
        by="PredictedPosition",
        ascending=True
    )
    

comparison["PredictedRank"] = comparison.groupby("Season")["PredictedPosition"].rank(
    method="first",
    ascending=True
).astype(int)

pred_error = (comparison["PredictedRank"] - comparison["Pos"]).abs()

for season, season_table in comparison.groupby("Season"):
    print(f"\nSeason: {season}")

    season_table = season_table.sort_values("Pos")

    season_table = season_table.copy()
    season_table["pred_error"] = pred_error.loc[season_table.index]
    print(
        season_table[["Team", "PredictedRank", "Pos", "pred_error"]]
        .to_string(index=False)
    )


Season: 2001-02

Season: 2002-03

Season: 2003-04

Season: 2004-05

Season: 2005-06

Season: 2006-07

Season: 2007-08

Season: 2008-09

Season: 2009-10

Season: 2010-11

Season: 2011-12

Season: 2012-13

Season: 2013-14

Season: 2014-15

Season: 2015-16

Season: 2016-17

Season: 2017-18

Season: 2018-19

Season: 2019-20

Season: 2020-21

Season: 2021-22

Season: 2022-23

Season: 2023-24

Season: 2024-25

Season: 2001-02
             Team  PredictedRank  Pos  pred_error
          Arsenal              3    1           2
        Liverpool              1    2           1
Manchester United              2    3           1
 Newcastle United              6    4           2
     Leeds United              4    5           1
          Chelsea              5    6           1
  West Ham United              8    7           1
      Aston Villa              7    8           1
Tottenham Hotspur             11    9           2
      Southampton             10   11           1
    Middlesbrough        